In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import IsolationForest
import os
import joblib
import warnings
warnings.filterwarnings("ignore")

# ========================= CONFIG (tune these) =========================
CHANNELS = list(range(41, 47))
WINDOW_SIZE = 50
DECISION_THRESHOLD = -0.15          # Lower = more anomalies (try -0.1, -0.2, -0.3)
CONTAMINATION = 0.015               # from previous

CACHE_DIR = "cache"

print("=== Windowed Isolation Forest - Adjusted Threshold ===\n")

# Load cached data and model
cache_file = os.path.join(CACHE_DIR, f"preprocessed_windowed_train_w{WINDOW_SIZE}.pkl")
X_train, df_train, scaler = joblib.load(cache_file)
print("Train data loaded from cache.")

model_file = os.path.join(CACHE_DIR, f"windowed_isoforest_w{WINDOW_SIZE}_cont{CONTAMINATION}.pkl")
model = joblib.load(model_file)
print("Model loaded from cache.")

# ========================= TEST INFERENCE =========================
print("Loading test.parquet...")
df_test = pd.read_parquet("/Users/alex/Downloads/esa-adb-challenge (1)/test.parquet")

channel_cols = [f'channel_{i}' for i in CHANNELS]
df_test = df_test[['id'] + channel_cols].copy()
df_test = df_test.sort_values('id').reset_index(drop=True)
df_test[channel_cols] = df_test[channel_cols].ffill().bfill()

test_data = scaler.transform(df_test[channel_cols].values)

# Create sliding windows
print(f"Creating test windows of size {WINDOW_SIZE}...")
X_test = []
for i in range(len(test_data) - WINDOW_SIZE + 1):
    window = test_data[i:i + WINDOW_SIZE].flatten()
    X_test.append(window)
X_test = np.array(X_test)

print("Computing scores...")
scores = model.decision_function(X_test)

# Apply tunable threshold
raw_anomalies = (scores < DECISION_THRESHOLD).astype(int)

print(f"Raw anomalies: {raw_anomalies.sum()} ({raw_anomalies.mean():.4%})")

# ========================= SIMPLE POST-PROCESSING =========================
print("Applying post-processing...")
anomaly_series = pd.Series(raw_anomalies)

# Merge detections within 5 points
anomaly_series = anomaly_series.rolling(window=5, min_periods=1, center=True).max().astype(int)

# Remove very short isolated detections (< 3 points)
anomaly_series = anomaly_series.rolling(window=3, min_periods=1, center=True).max().astype(int)

clean_anomalies = anomaly_series.values

print(f"Clean anomalies after post-processing: {clean_anomalies.sum()} ({clean_anomalies.mean():.4%})")

# ========================= SUBMISSION =========================
submission = pd.DataFrame({
    'id': df_test['id'].iloc[WINDOW_SIZE-1:].reset_index(drop=True),
    'anomaly_flag': clean_anomalies
})

submission.to_parquet("submission_windowed_isoforest_adjusted.parquet", index=False)

print("\n✅ Submission saved as 'submission_windowed_isoforest_adjusted.parquet'")
print("Upload this to Kaggle Late Submission to see the real F0.5 score.")

Loading preprocessed TRAIN from cache...
Loading trained model from cache...

Loading test.parquet for inference...
Computing anomaly scores on test set...

✅ Submission saved! Shape: (521280, 2)
         id  is_anomaly
0  14728321           0
1  14728322           0
2  14728323           0
3  14728324           0
4  14728325           0
Raw anomalies: 0 (0.0000%)
Clean anomalies: 0 (0.0000%)
